# Laytime per cargo port (`CargoPortLaytime`)

Exercises `CargoPortLaytime` in `models.py` and `apply_laytime_durations()` in `calculate.py`.

- **Resolver / LLM** supplies timestamps, stoppages, and evidence — not `gross_duration_seconds`, `deducted_duration_seconds`, or `used_duration_seconds`.
- **Python** fills those seconds and checks `used = gross − deducted` on the model.
- No API key; uses the same voyage windows as `tests/test_skeleton.py`.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

from pydantic import ValidationError

from src.calculate import apply_laytime_durations, compute_demurrage
from src.models import CargoPortLaytime, ContractTerms, EvidenceRef, StoppageDeduction
from src.resolver import itinerary_skeleton
from src.validate import validate_resolved

VOYAGE = {
    "voyage_no": "KB-2603",
    "itinerary": [
        {"seq": 1, "port": "Antwerp", "function": "load"},
        {"seq": 2, "port": "Gibraltar", "function": "bunkering"},
        {"seq": 3, "port": "Cotonou", "function": "discharge"},
    ],
}

## Contract + resolver-shaped laytime rows

Only load/discharge visits get a `CargoPortLaytime` row (`itinerary_skeleton`).

In [ ]:
SID = "email_01.eml:abc"
QUOTE = "72 hours reversible, demurrage USD 25,500 pdpr"

contract = ContractTerms(
    status="resolved",
    laytime_allowance_hours=72,
    reversible=True,
    turn_time_after_nor_hours=6,
    demurrage_rate_per_day=25500,
    currency="USD",
    evidence=[EvidenceRef(source_id=SID, quote=QUOTE)],
)

skeleton = itinerary_skeleton(VOYAGE)
windows = {
    "pv-1": ("2026-03-01T06:00:00+00:00", "2026-03-02T20:00:00+00:00"),
    "pv-3": ("2026-03-20T00:00:00+00:00", "2026-03-23T23:30:00+00:00"),
}

resolver_rows = [
    lt.model_copy(
        update={
            "laytime_start": windows[lt.visit_id][0],
            "final_hoses_disconnected": windows[lt.visit_id][1],
            "status": "final",
        }
    )
    for lt in skeleton.laytime_by_cargo_port
]

for lt in resolver_rows:
    print(lt.visit_id, lt.port, lt.laytime_start, "→", lt.final_hoses_disconnected)
    print("  durations (should be null):", lt.gross_duration_seconds, lt.deducted_duration_seconds, lt.used_duration_seconds)

## `apply_laytime_durations` → filled `CargoPortLaytime`

In [ ]:
filled = apply_laytime_durations(contract, resolver_rows)

def hours(seconds):
    return None if seconds is None else seconds / 3600

print(f"{'visit':<6} {'port':<10} {'gross h':>8} {'ded h':>8} {'used h':>8}")
for lt in filled:
    print(
        f"{lt.visit_id:<6} {lt.port:<10} "
        f"{hours(lt.gross_duration_seconds):>8.1f} "
        f"{hours(lt.deducted_duration_seconds):>8.1f} "
        f"{hours(lt.used_duration_seconds):>8.1f}"
    )

demurrage = compute_demurrage(contract, filled)
print()
print("total used (h):", hours(demurrage.total_used_seconds))
print("allowed (h):", hours(demurrage.allowed_seconds))
print("excess (h):", hours(demurrage.excess_seconds))
print("amount owed:", demurrage.currency, demurrage.amount_owed)

## Stoppages reduce `used` (DOMAIN: used = gross − deductions)

Add a 2 h vessel stoppage at Antwerp; `duration_seconds` on each `StoppageDeduction` is what Python sums today.

In [ ]:
with_stoppage = resolver_rows[0].model_copy(
    update={
        "stoppages": [
            StoppageDeduction(
                description="Pump failure",
                duration_seconds=2 * 3600,
                contract_rule="TIME LOST DUE TO BREAKDOWN ... NOT TO COUNT",
            )
        ]
    }
)
rows = [with_stoppage, resolver_rows[1]]
filled2 = apply_laytime_durations(contract, rows)
lt = filled2[0]
print("Antwerp gross (h):", lt.gross_duration_seconds / 3600)
print("deducted (h):", lt.deducted_duration_seconds / 3600)
print("used (h):", lt.used_duration_seconds / 3600)
assert lt.used_duration_seconds == lt.gross_duration_seconds - lt.deducted_duration_seconds

## Model guardrails

- Pydantic rejects `used ≠ gross − deducted` when all three are set.
- `validate_resolved` rejects duration seconds on resolver output before calculation.

In [ ]:
try:
    CargoPortLaytime(
        visit_id="pv-1",
        port="Antwerp",
        gross_duration_seconds=10,
        deducted_duration_seconds=2,
        used_duration_seconds=9,
    )
except ValidationError as e:
    print("Pydantic:", e.errors()[0]["msg"])

from src.models import EmailExtraction

bad = filled[0].model_copy(update={"gross_duration_seconds": 1})
core = skeleton.model_copy(
    update={"contract": contract, "laytime_by_cargo_port": [bad, filled[1]]}
)
extractions = [
    EmailExtraction(
        source_id=SID,
        filename="email_01.eml",
        content_hash="abc",
        message_id=None,
        subject="CP",
        sent_at=None,
        extraction_status="complete",
        voyage_about="yes",
        vessel_name="KESTREL BAY",
        relevance="relevant",
        topics={
            "contract": "yes",
            "schedule": "no",
            "laytime": "no",
            "position": "no",
            "other": "no",
        },
    )
]
problems = validate_resolved(core, VOYAGE, extractions)
print("validate_resolved:", problems)

In [ ]:
print(CargoPortLaytime.model_json_schema())